In [2]:
import pandas as pd

PARQUET_PATH = "data/train/omni_dataset_gt2000.parquet"

df = pd.read_parquet(PARQUET_PATH)

for i, thought in enumerate(df["thought"].head(20), 1):
    print(f"\n===== ROW {i} =====")
    print(thought)


===== ROW 1 =====
The problem states: "3. Answer: \( C_{12}^{3} = 220 \)". It seems incomplete. I think it's referring to a combinatorial problem where we need to find the number of ways to choose 3 items out of 12, and the answer is given as 220. But without the actual question, it's hard to know what's being asked. Perhaps the question is implied or missing. Looking back, it says "3. Answer: \( C_{12}^{3} = 220 \)", so probably the question is to compute \( C_{12}^{3} \), and the answer is provided. But that doesn't make much sense for a problem. Maybe it's part of a larger context.

Perhaps the "3." is the problem number, and "Answer: \( C_{12}^{3} = 220 \)" is the answer, but I need to find the question or verify it. The user might want me to explain how this answer is obtained. That could be it. So, I'll assume that the task is to compute the combination \( C_{12}^{3} \), which is the number of ways to choose 3 items from 12 without regard to order.

The formula for combinations 

In [1]:
import os
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer
from jinja2 import Environment, FileSystemLoader

MODEL_PATH = "Qwen/Qwen3-0.6B"  # hoặc path local
TEMPLATE_PATH = "configs/chat_template/chat_template.jinja"

# =========================
# LOAD TOKENIZER + TEMPLATE
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

template_dir = os.path.dirname(TEMPLATE_PATH)
template_file = os.path.basename(TEMPLATE_PATH)

env = Environment(loader=FileSystemLoader(template_dir))
template = env.get_template(template_file)

# system_prompt = "Solve problems step-by-step within <think> tags. You must analyze constraints, verify each step, and actively self-correct errors before providing the final answer."
system_prompt = "You are a careful mathematical reasoning assistant. Before responding, reason in <think>...</think>. Use it to understand the task, identify relevant constraints, reason logically, verify key steps, and correct mistakes when found. Then respond clearly and follow the user's instructions."
# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# 3. Setup Template (Phần code của bạn)
template_path = TEMPLATE_PATH
template_dir = os.path.dirname(template_path)
template_file = os.path.basename(template_path)
env = Environment(loader=FileSystemLoader(template_dir))
template = env.get_template(template_file)

messages = [
    {"role": "system", "content": system_prompt},
    # {"role": "user", "content": "QUESTION"},
    # {
    #     "role": "assistant",
    #     "reasoning_content": "THOUGHT",
    #     "content": "SOLUTION",
    # },
]

text = template.render(
    messages=messages,
    add_generation_prompt=False,
)

text

outputs = tokenizer(
    text,
    add_special_tokens=False,
    padding=False,
    truncation=False,
)
# 6. ĐẾM TOKEN (Sửa lỗi ở đây)
# outputs["input_ids"] là một list chứa các ID của token
token_count = len(outputs["input_ids"])

print(f"Nội dung sau khi render:\n{text}")
print(f"---")
print(f"Số lượng token: {token_count}")

c:\Users\admin\miniconda3\envs\thesis\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Nội dung sau khi render:
<|im_start|>system
You are a careful mathematical reasoning assistant. Before responding, reason in <think>...</think>. Use it to understand the task, identify relevant constraints, reason logically, verify key steps, and correct mistakes when found. Then respond clearly and follow the user's instructions.<|im_end|>

---
Số lượng token: 57


In [ ]:
print(text)

<|im_start|>system
Solve problems step-by-step within <think> tags. You must analyze constraints, verify each step, and actively self-correct errors before providing the final answer.<|im_end|>



In [24]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer
from tqdm import tqdm

# =========================
# CONFIGURATION
# =========================
MODEL_PATH = "Qwen/Qwen3-0.6B"  # Thay bằng path local hoặc model hub
PARQUET_PATH = "data/train/omni_dataset_le2000.parquet"   # Đường dẫn tới file parquet của bạn
SYSTEM_PROMPT = "Solve problems step-by-step within <think> tags. You must analyze constraints, verify each step, and actively self-correct errors before providing the final answer."

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# 2. Load Parquet (Question, Thought, Solution)
df = pd.read_parquet(PARQUET_PATH)

token_counts = []

print(f"Processing {len(df)} rows...")

# 3. Duyệt qua từng hàng để render và đếm
for _, row in tqdm(df.iterrows(), total=len(df)):
    # Chuẩn bị cấu trúc tin nhắn cho Chat Template
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["question"]},
        {
            "role": "assistant",
            "reasoning_content": row['thought'],
            "content": row['solution'],
        }
    ]
    
    # Render ra text thô dùng Chat Template của Model
    # add_generation_prompt=False vì chúng ta đã có câu trả lời của assistant
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=False
    )
    
    # Encode text đã render để đếm token
    outputs = tokenizer(
            text,
            add_special_tokens=False,  # template đã tự thêm special tokens nếu có
            padding=False,
            truncation=False,
        )
    count = len(outputs["input_ids"])
    
    token_counts.append(count)

# 4. Tính toán thống kê
if token_counts:
    max_tokens = max(token_counts)
    min_tokens = min(token_counts)
    mean_tokens = np.mean(token_counts)
    total_tokens = sum(token_counts)

    print("\n" + "="*30)
    print(f"TOKEN STATISTICS ({MODEL_PATH})")
    print("="*30)
    print(f"Total Rows   : {len(df)}")
    print(f"Total Tokens : {total_tokens:,}")
    print(f"Max Tokens   : {max_tokens}")
    print(f"Min Tokens   : {min_tokens}")
    print(f"Mean Tokens  : {mean_tokens:.2f}")
    print("="*30)
else:
    print("No data found to process.")

Processing 10000 rows...


100%|██████████| 10000/10000 [00:24<00:00, 407.34it/s]


TOKEN STATISTICS (Qwen/Qwen3-0.6B)
Total Rows   : 10000
Total Tokens : 15,738,840
Max Tokens   : 2035
Min Tokens   : 1035
Mean Tokens  : 1573.88


In [3]:
import pandas as pd

PARQUET_PATH = "data/train/omni_dataset_gt2000_sub12k.parquet"

df = pd.read_parquet(PARQUET_PATH)

df.columns

Index(['question', 'solution', 'thought', 'final_answer', 'verbosity',
       'difficulty', 'is_math', 'total_token', 'train_token', 'index'],
      dtype='object')